In [ ]:
# try: merge base model with speech-lora, treat it as new base mode, load using vllm

# F1: vllm supports switching between multiple LoRAs, but does not support stacking

# Q1: When vllm loads, is the speech LoRA loaded separately?
# A1: Phi-4-MM contains three LoRAs: base_layer, speech, vision. The base_layer is loaded by default.
# Q2: Can we just modify it to load speech by default? <=> Does the original model's set_adapters just switch directly?
# A2: No, base_layer is the fundamental layer of PEFT, all embeddings must pass through it. We need to merge the model, specifically, merge speech into base_layer, save it to get the new weights, and then use vllm to load this speech_base_model.

In [1]:
from huggingface_hub import snapshot_download
from safetensors.torch import load_file
import torch


path = snapshot_download("microsoft/Phi-4-multimodal-instruct")
path

Fetching 54 files:   0%|          | 0/54 [00:00<?, ?it/s]

'/home/aiscuser/.cache/huggingface/hub/models--microsoft--Phi-4-multimodal-instruct/snapshots/33e62acdd07cd7d6635badd529aa0a3467bb9c6a'

In [ ]:
!ls $path

In [ ]:
weight = load_file(path + "/model-00001-of-00003.safetensors")
weight.keys()

In [2]:
from transformers import AutoModelForCausalLM

model = AutoModelForCausalLM.from_pretrained("microsoft/Phi-4-multimodal-instruct", trust_remote_code=True)
model

/bin/ld: cannot find -laio: No such file or directory
collect2: error: ld returned 1 exit status
/bin/ld: cannot find -laio: No such file or directory
collect2: error: ld returned 1 exit status
/usr/local/lib/python3.12/dist-packages/transformers/models/auto/image_processing_auto.py:604: FutureWarning: The image_processor_class argument is deprecated and will be removed in v4.42. Please use `slow_image_processor_class`, or `fast_image_processor_class` instead
  warnings.warn(
You are attempting to use Flash Attention 2.0 without specifying a torch dtype. This might lead to unexpected behaviour
You are attempting to use Flash Attention 2.0 with a model not initialized on GPU. Make sure to move the model to GPU after initializing it on CPU with `model.to('cuda')`.
/home/aiscuser/.cache/huggingface/modules/transformers_modules/microsoft/Phi-4-multimodal-instruct/33e62acdd07cd7d6635badd529aa0a3467bb9c6a/speech_conformer_encoder.py:2774: FutureWarning: Please specify CheckpointImpl.NO_REENT

Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

Phi4MMForCausalLM(
  (model): Phi4MMModel(
    (embed_tokens): Embedding(200064, 3072, padding_idx=199999)
    (embed_dropout): Dropout(p=0.0, inplace=False)
    (embed_tokens_extend): Phi4MMImageAudioEmbedding(
      (image_embed): Phi4MMImageEmbedding(
        (drop): Dropout(p=0.0, inplace=False)
        (img_processor): SiglipVisionTransformer(
          (embeddings): SiglipVisionEmbeddings(
            (patch_embedding): Conv2d(3, 1152, kernel_size=(14, 14), stride=(14, 14), padding=valid)
            (position_embedding): Embedding(1024, 1152)
          )
          (encoder): SiglipEncoder(
            (layers): ModuleList(
              (0-26): 27 x SiglipEncoderLayer(
                (self_attn): SiglipFlashAttention2(
                  (k_proj): Linear(in_features=1152, out_features=1152, bias=True)
                  (v_proj): Linear(in_features=1152, out_features=1152, bias=True)
                  (q_proj): Linear(in_features=1152, out_features=1152, bias=True)
              

In [3]:
from peft.tuners.lora.layer import LoraLayer
for module in model.modules():
    if isinstance(module, LoraLayer):
        module.merge(adapter_names=["speech"])
        module.merged_adapters = []


In [ ]:
model

In [4]:
model = model.to(torch.bfloat16)

In [5]:
model.save_pretrained(
    "/mnt/blob/v-chaorwang/Phi-4-MM-speech", safe_serialization=True
)

In [ ]:

state = model.state_dict()
total_bytes = 0
for k, v in state.items():
    print(k, v.dtype, v.numel())
    total_bytes += v.numel() * v.element_size()
print("total MB:", total_bytes / 1024**2)

In [ ]:
del model

In [ ]:
!echo $path

In [ ]:
!cp /home/aiscuser/.cache/huggingface/hub/models--microsoft--Phi-4-multimodal-instruct/snapshots/33e62acdd07cd7d6635badd529aa0a3467bb9c6a/{merges.txt,preprocessor_config.json,processor_config.json,special_tokens_map.json,tokenizer.json,tokenizer_config.json,vocab.json} /scratch/phi-4-mm-speech

In [ ]:
!cp modeling_phi4mm.py /scratch/phi-4-mm-speech

In [1]:
from vllm.utils import LRUCache
def hooked_touch(self, key):
    try:
        self._LRUCache__order.move_to_end(key)
    except KeyError:
        self._LRUCache__order[key] = None
LRUCache.touch = hooked_touch

INFO 11-04 08:00:33 [__init__.py:239] Automatically detected platform cuda.


In [2]:
from vllm import LLM

llm = LLM(
    "/scratch/phi-4-mm-speech", 
    enable_lora=True, 
    trust_remote_code=True, 
    max_model_len=12800,
    max_num_seqs=2,
    max_lora_rank=320,
    limit_mm_per_prompt={"audio": 1},
    gpu_memory_utilization=0.4,
    device="cuda"
)
llm

INFO 11-04 08:00:35 [config.py:209] Replacing legacy 'type' key with 'rope_type'


INFO 11-04 08:00:50 [config.py:600] This model supports multiple tasks: {'score', 'generate', 'reward', 'classify', 'embed'}. Defaulting to 'generate'.
WARNING 11-04 08:00:50 [arg_utils.py:1708] ['Phi4MMForCausalLM'] is not supported by the V1 Engine. Falling back to V0. 
INFO 11-04 08:00:50 [llm_engine.py:242] Initializing a V0 LLM engine (v0.8.3) with config: model='/scratch/phi-4-mm-speech', speculative_config=None, tokenizer='/scratch/phi-4-mm-speech', skip_tokenizer_init=False, tokenizer_mode=auto, revision=None, override_neuron_config=None, tokenizer_revision=None, trust_remote_code=True, dtype=torch.bfloat16, max_seq_len=12800, download_dir=None, load_format=LoadFormat.AUTO, tensor_parallel_size=1, pipeline_parallel_size=1, disable_custom_all_reduce=False, quantization=None, enforce_eager=False, kv_cache_dtype=auto,  device_config=cuda, decoding_config=DecodingConfig(guided_decoding_backend='xgrammar', reasoning_backend=None), observability_config=ObservabilityConfig(show_hidden

[W1104 08:00:53.087201426 socket.cpp:759] [c10d] The client socket cannot be initialized to connect to [100-64-33-234.node-0.acf32e4e-29c6-48b3-8d5b-ca84f2402aef.svc.cluster.local]:53191 (errno: 97 - Address family not supported by protocol).


Loading safetensors checkpoint shards:   0% Completed | 0/3 [00:00<?, ?it/s]


INFO 11-04 08:00:56 [loader.py:447] Loading weights took 2.42 seconds
WARNING 11-04 08:00:56 [model_runner.py:1120] Regarding multimodal models, vLLM currently only supports adding LoRA to language model.
INFO 11-04 08:00:56 [punica_selector.py:18] Using PunicaWrapperGPU.
WARNING 11-04 08:00:56 [models.py:480] Regarding multimodal models, vLLM currently only supports adding LoRA to language model, vision_encoder.img_processor.encoder.layers.0.self_attn.qkv_proj will be ignored.
WARNING 11-04 08:00:56 [models.py:480] Regarding multimodal models, vLLM currently only supports adding LoRA to language model, vision_encoder.img_processor.encoder.layers.0.self_attn.out_proj will be ignored.
WARNING 11-04 08:00:56 [models.py:480] Regarding multimodal models, vLLM currently only supports adding LoRA to language model, vision_encoder.img_processor.encoder.layers.0.mlp.fc1 will be ignored.
WARNING 11-04 08:00:56 [models.py:480] Regarding multimodal models, vLLM currently only supports adding LoRA

Capturing CUDA graph shapes: 100%|██████████| 2/2 [00:02<00:00,  1.11s/it]

INFO 11-04 08:01:08 [model_runner.py:1598] Graph capturing finished in 4 secs, took 0.07 GiB
INFO 11-04 08:01:08 [llm_engine.py:448] init engine (profile, create kv cache, warmup model) took 11.26 seconds


In [33]:
import torchaudio
from vllm.lora.request import LoRARequest
from vllm import LLM, SamplingParams

audio = torchaudio.load("/mnt/blob/v-chaorwang/MMLU_FISH/question_wav.v3/mmlu@en_4944.wav")
audio, sr = audio

# inputs = ["<|audio_1|>Transcribe the speech."]
inputs = ["<|audio_1|>"]
audios = [(audio[0], sr)]

user_prompt = "<|user|>"
assistant_prompt = "<|assistant|>"
prompt_suffix = "<|end|>"

temperature = 0.0
max_new_tokens = 1024

In [34]:
lora = [None]

In [ ]:
lora = [LoRARequest("speech", 1, "/home/aiscuser/.cache/huggingface/hub/models--microsoft--Phi-4-multimodal-instruct/snapshots/33e62acdd07cd7d6635badd529aa0a3467bb9c6a/speech-lora")] # should not work..., and yes! repetition occured.

In [ ]:
lora = [LoRARequest("speech2", 2, "/scratch/phi4-dpo/OUTPUT_GRPO/v12_cotrwAccEmbed_dapo_mixm2_e1_dsftcotv1_bs32_20251104_0207/checkpoint-1000")] # success to solve Q4944

In [43]:
inputs_vllm = []
texts = []
for i, input in enumerate(inputs):
    texts.append(f"{user_prompt}{input}{prompt_suffix}{assistant_prompt}")
    inputs_vllm.append(
        {
            "prompt": texts[i],
            "multi_modal_data": {"audio": audios[i]} if audios is not None else None,
        }
    )

sampling_params = SamplingParams(temperature=temperature, max_tokens=max_new_tokens)

lora = lora * len(inputs_vllm)

outputs = llm.generate(
    inputs_vllm,
    sampling_params=sampling_params,
    lora_request=lora,
)
outputs

Processed prompts: 100%|██████████| 1/1 [00:03<00:00,  3.63s/it, est. speed input: 110.45 toks/s, output: 30.30 toks/s]


[RequestOutput(request_id=10, prompt='<|user|><|audio_1|><|end|><|assistant|>', prompt_token_ids=[200021, 200011, 200011, 200011, 200011, 200011, 200011, 200011, 200011, 200011, 200011, 200011, 200011, 200011, 200011, 200011, 200011, 200011, 200011, 200011, 200011, 200011, 200011, 200011, 200011, 200011, 200011, 200011, 200011, 200011, 200011, 200011, 200011, 200011, 200011, 200011, 200011, 200011, 200011, 200011, 200011, 200011, 200011, 200011, 200011, 200011, 200011, 200011, 200011, 200011, 200011, 200011, 200011, 200011, 200011, 200011, 200011, 200011, 200011, 200011, 200011, 200011, 200011, 200011, 200011, 200011, 200011, 200011, 200011, 200011, 200011, 200011, 200011, 200011, 200011, 200011, 200011, 200011, 200011, 200011, 200011, 200011, 200011, 200011, 200011, 200011, 200011, 200011, 200011, 200011, 200011, 200011, 200011, 200011, 200011, 200011, 200011, 200011, 200011, 200011, 200011, 200011, 200011, 200011, 200011, 200011, 200011, 200011, 200011, 200011, 200011, 200011, 200011

In [44]:
outputs[0].outputs[0].text, outputs[0].lora_request.lora_path

("John's inability to remember the pictures on the coins and bills is most likely due to a failure to encode. Encoding is the process of converting information into a form that can be stored in memory. Since John has been handling money for 17 years, he has likely developed a strong familiarity with the physical appearance of the coins and bills. However, he is unable to recall the specific images associated with each denomination, which suggests that the information has not been effectively encoded into his memory. Therefore, the correct answer is (C) failure to encode.",
 '/scratch/phi4-dpo/OUTPUT_GRPO/v12_cotrwAccEmbed_dapo_mixm2_e1_dsftcotv1_bs32_20251104_0207/checkpoint-1000')